# Generating QRC inputs from an ASE / MLIP frequency calculation

pyQRC normally reads a Gaussian, ORCA, or Q-Chem output file. When the Hessian comes from a machine-learned interatomic potential (MLIP) driven through [ASE](https://wiki.fysik.dtu.dk/ase/) there is no such file — so the helper [`ase2gaussian.py`](ase2gaussian.py) in this directory writes the geometry, frequencies, and normal modes in the Gaussian log format that cclib (and therefore pyQRC) parses. From there, everything works exactly as with a real QM frequency job.

This notebook uses the [MACE-OFF](https://github.com/ACEsuit/mace) foundation model for organic molecules, but any ASE calculator that supports forces works the same way (ANI, AIMNet2, xTB, SO3LR, ...).

Requirements: `pip install pyqrc ase mace-torch`

## Setup

We work in a `scratch/` subdirectory (ignored by git) since pyQRC writes its files next to the log it reads. The thread settings avoid an OpenMP crash with PyTorch on macOS and must come before the first `torch` import.

In [1]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import shutil
import subprocess
import sys
from pathlib import Path

import numpy as np
from ase import Atoms
from ase.constraints import FixedPlane
from ase.io import read
from ase.optimize import LBFGS
from ase.vibrations import Vibrations

sys.path.insert(0, str(Path.cwd()))
from ase2gaussian import extract_vibrations, write_gaussian_freq_log

HERE = Path.cwd()
SCRATCH = HERE / "scratch"
if SCRATCH.exists():
    shutil.rmtree(SCRATCH)
SCRATCH.mkdir()
os.chdir(SCRATCH)

EV_TO_HARTREE = 1 / 27.211386245988
EV_TO_KCAL = 23.0605


def run_pyqrc(*args):
    """Run the pyqrc command line inside scratch/, echoing its output."""
    result = subprocess.run(
        [sys.executable, "-m", "pyqrc", *args],
        capture_output=True, text=True, cwd=SCRATCH,
    )
    print(result.stdout, end="")
    if result.returncode != 0:
        print(result.stderr, end="")
    return result.returncode


def show(filename, n=None):
    """Print the first n lines (default: all) of a generated file."""
    lines = (SCRATCH / filename).read_text().splitlines()
    print("\n".join(lines if n is None else lines[:n]))

We use one calculator for the whole notebook: MACE-OFF (medium), the recommended general-purpose model for organic molecules.

In [2]:
from mace.calculators import mace_off

calc = mace_off(model="medium", default_dtype="float64", device="cpu")

cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.
Using MACE-OFF23 MODEL for MACECalculator with /Users/rpaton/.cache/mace/MACE-OFF23_medium.model
Using float64 for MACECalculator, which is slower but more accurate. Recommended for geometry optimization.


## Example 1: NH$_3$ inversion — from a saddle point to the minimum

Planar NH$_3$ is the transition state for umbrella inversion. We optimize it with LBFGS under a planarity constraint (a poor man's saddle-point search — for real systems use a TS optimizer like Sella), then release the constraint for the frequency calculation.

In [3]:
nh3 = Atoms("NH3", positions=[[0.00, 0.000, 0.0],
                              [1.02, 0.000, 0.0],
                              [-0.51, 0.883, 0.0],
                              [-0.51, -0.883, 0.0]])
nh3.calc = calc
nh3.set_constraint([FixedPlane(i, (0, 0, 1)) for i in range(len(nh3))])
LBFGS(nh3, logfile=None).run(fmax=0.001)
nh3.set_constraint()
e_ts = nh3.get_potential_energy()
print(f"planar NH3: E = {e_ts:.6f} eV, N-H = {nh3.get_distance(0, 1):.4f} A")

planar NH3: E = -1539.862552 eV, N-H = 0.9961 A


`ase.vibrations.Vibrations` builds the Hessian by finite differences of the MLIP forces. `extract_vibrations` converts ASE's complex energies to the Gaussian sign convention (negative = imaginary) and drops the six near-zero translation/rotation modes.

In [4]:
vib = Vibrations(nh3, name="vib_nh3")
vib.run()
frequencies, modes = extract_vibrations(vib)
print("frequencies (cm-1):", np.round(frequencies, 1))

frequencies (cm-1): [-824.3 1538.8 1539.2 3640.6 3847.4 3847.8]


Exactly one imaginary frequency — the umbrella inversion mode — as expected for a first-order saddle point. `write_gaussian_freq_log` emits only the blocks cclib and pyQRC actually read; the `route` argument (default `# opt`) is echoed into the inputs pyQRC generates.

In [5]:
write_gaussian_freq_log("nh3_ts_mace.log", nh3, frequencies, modes,
                        energy=e_ts * EV_TO_HARTREE)
show("nh3_ts_mace.log", n=22)

 Entering Gaussian System, Link 0=g16
 This file was written by ase2gaussian.py (pyQRC examples), not by
 Gaussian. It mimics the output format of Gaussian, Inc. so that
 cclib-based tools can read ASE/MLIP frequency results.
 ----------------------------------------------------------------------
 # opt
 ----------------------------------------------------------------------
 Charge =  0 Multiplicity = 1
 NAtoms=     4 NActive=     4
                         Standard orientation:                         
 ---------------------------------------------------------------------
 Center     Atomic      Atomic             Coordinates (Angstroms)
 Number     Number       Type             X           Y           Z
 ---------------------------------------------------------------------
      1          7           0        0.000038    0.000000    0.000000
      2          1           0        0.996152    0.000000    0.000000
      3          1           0       -0.498095    0.862598    0.000000
 

By default pyQRC displaces along all imaginary modes — here, the single inversion mode. The benchmark in the README found an amplitude of **0.3** performs best.

In [6]:
run_pyqrc("nh3_ts_mace.log", "--amp", "0.3", "--name", "QRC")
show("nh3_ts_mace_QRC.com")

o   nh3_ts_mace.log has 1 imaginary frequencies: processing
%chk=nh3_ts_mace_QRC.chk
%nproc=1
%mem=4GB
# opt


nh3_ts_mace_QRC

0 1
 N   0.00003800   0.00000000   0.03600000
 H   0.99615200   0.00000000  -0.17100000
 H  -0.49809500   0.86259800  -0.17100000
 H  -0.49809500  -0.86259800  -0.17100000



The nitrogen has moved out of the H$_3$ plane. In a QM workflow you would now submit this `.com` file to Gaussian. But since our potential is an MLIP, we can close the loop right here: read the displaced geometry back into ASE and let LBFGS relax it into the adjacent minimum.

In [7]:
displaced = read("nh3_ts_mace_QRC.com", format="gaussian-in")
displaced.calc = calc
LBFGS(displaced, logfile=None).run(fmax=0.001)
e_min = displaced.get_potential_energy()

n_height = np.linalg.norm(displaced.positions[0] - displaced.positions[1:].mean(axis=0))
print(f"N out-of-plane distance: {n_height:.3f} A (0 = planar)")
print(f"inversion barrier: {(e_ts - e_min) * EV_TO_KCAL:.1f} kcal/mol")

N out-of-plane distance: 0.376 A (0 = planar)
inversion barrier: 4.8 kcal/mol


The structure relaxed to pyramidal NH$_3$ with an inversion barrier close to the experimental ~5.8 kcal/mol — the whole minimum-finding loop ran on the MLIP without a single QM calculation.

## Example 2: map the Claisen reaction coordinate from a DFT transition state

The [g16 example directory](../g16/) contains a DFT transition state for the Claisen rearrangement of allyl vinyl ether ([claisen_ts.log](../g16/claisen_ts.log)). Suppose that QM job is all you have — an MLIP Hessian and MLIP relaxations can map out which reactant and product the TS connects without any further QM.

One honest caveat: the DFT geometry is *not* a stationary point on the MLIP surface, so the residual forces contaminate the low-frequency end of the spectrum (the trans/rot modes are not cleanly zero, and the imaginary frequency is exaggerated relative to the DFT value of 590i cm$^{-1}$). That does not matter for QRC — the reaction-coordinate mode is far stronger than the noise, and we displace along it alone with `--freqnum 1`.

In [8]:
ts = read(HERE.parent / "g16" / "claisen_ts.log", format="gaussian-out")
ts.calc = calc
e_ts = ts.get_potential_energy()

vib = Vibrations(ts, name="vib_claisen")
vib.run()
frequencies, modes = extract_vibrations(vib)
print("lowest modes (cm-1):", np.round(frequencies[:6], 1))

write_gaussian_freq_log("claisen_ts_mace.log", ts, frequencies, modes,
                        energy=e_ts * EV_TO_HARTREE)

lowest modes (cm-1): [-1226.7   247.3   286.    295.1   338.8   388.7]


As in README Example 2, the quick alternative to an IRC is two displaced inputs — one along the imaginary mode, one in the reverse direction:

In [9]:
run_pyqrc("claisen_ts_mace.log", "--amp", "0.3", "--freqnum", "1", "--name", "QRCF")
run_pyqrc("claisen_ts_mace.log", "--amp", "-0.3", "--freqnum", "1", "--name", "QRCR")

o   claisen_ts_mace.log will be distorted along freq #1: processing
o   claisen_ts_mace.log will be distorted along freq #1: processing


0

Relax both displaced geometries with the MLIP and identify what they became from the carbon-oxygen distances: a C-O-C ether (two C-O bonds of ~1.4 A) is the reactant, a C=O carbonyl (one short ~1.2 A bond) is the product.

In [10]:
def shortest_co_distances(atoms, n=2):
    symbols = atoms.get_chemical_symbols()
    oxygen = symbols.index("O")
    dists = sorted(atoms.get_distance(oxygen, j)
                   for j, s in enumerate(symbols) if s == "C")
    return dists[:n]


energies = {}
for name in ["QRCF", "QRCR"]:
    geom = read(f"claisen_ts_mace_{name}.com", format="gaussian-in")
    geom.calc = calc
    LBFGS(geom, logfile=None).run(fmax=0.005)
    energies[name] = geom.get_potential_energy()
    d1, d2 = shortest_co_distances(geom)
    identity = ("4-pentenal (C=O product)" if d2 > 1.8
                else "allyl vinyl ether (C-O-C reactant)")
    print(f"{name}: C-O = {d1:.2f}, {d2:.2f} A -> {identity}")

print(f"\nreaction energy: {(energies['QRCF'] - energies['QRCR']) * EV_TO_KCAL:.1f} kcal/mol")

QRCF: C-O = 1.20, 2.40 A -> 4-pentenal (C=O product)
QRCR: C-O = 1.36, 1.43 A -> allyl vinyl ether (C-O-C reactant)

reaction energy: -19.0 kcal/mol


The two QRC displacements land in the reactant and product wells: allyl vinyl ether on one side, 4-pentenal on the other, with an exothermicity of ~19 kcal/mol — in line with the experimental Claisen reaction energy of roughly -17 to -20 kcal/mol. (Energies measured *from the TS* are not meaningful barriers here, because the unrelaxed DFT geometry sits above the true MLIP saddle.)

## Where the files went

Everything generated above is in the `scratch/` subdirectory — delete it when you are done. For your own systems, the recipe is: any ASE calculator → `Vibrations` → `extract_vibrations` → `write_gaussian_freq_log` → `pyqrc`, with all of pyQRC's options (`--amp`, `--freqnum`, `--auto`, reverse displacement via negative amplitudes) available as usual. See the [project README](../../README.md) for the full option list.